# CCTV 다중 identity ReID 버블차트

이 노트북은 단일 인물 파일럿을 일반 정확도로 사용하지 않고, CHIRLA 다중 identity gallery/query 계약에서 모델을 같은 조건으로 비교한다. x축은 Rank-1, y축은 mAP/MINP, 버블 크기는 Recall@10이다. 85% 기준선은 실제 승격 게이트이며, 현재 proxy 결과가 통과했는지 자동으로 표시한다.

주의: CHIRLA는 프로젝트 검수 CCTV가 아닌 공개 proxy다. 최종 배포 승격에는 프로젝트 identity·track·속성 라벨의 별도 sealed test가 필요하다.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path('../experiments/results/chirla_reid_comparison_20260724.json'), Path('experiments/results/chirla_reid_comparison_20260724.json')]
result_path = next(path for path in candidates if path.exists())
payload = json.loads(result_path.read_text(encoding='utf-8'))
df = pd.DataFrame(payload['results'])
df[['model', 'gallery_aggregation', 'rank1', 'map', 'recall_at_10', 'rank1_ci95_low', 'rank1_ci95_high']]

In [ ]:
plt.rcParams['font.family'] = ['Malgun Gothic', 'DejaVu Sans']
fig, ax = plt.subplots(figsize=(12, 7))
sizes = 500 + (df['recall_at_10'] * 1000)
ax.scatter(df['rank1'] * 100, df['map'] * 100, s=sizes, alpha=0.72, edgecolor='white', linewidth=1.2)
for _, row in df.iterrows():
    label = row['model'].replace('clip-vit-', 'CLIP ').replace('siglip2-base', 'SigLIP2').replace('dinov2-base', 'DINOv2').replace('osnet-x1-0', 'OSNet')
    if row['gallery_aggregation'] != 'mean':
        label += f"\n{row['gallery_aggregation']}"
    ax.annotate(label, (row['rank1'] * 100, row['map'] * 100), xytext=(6, 6), textcoords='offset points', fontsize=9)
ax.axvline(85, color='crimson', linestyle='--', linewidth=2, label='85% 승격 게이트')
ax.set(xlim=(0, 100), ylim=(0, 60), xlabel='Rank-1 정확도 (%)', ylabel='mAP / MINP (%)', title='CHIRLA 다중 identity 모델 비교')
ax.grid(alpha=0.22)
ax.legend()
plt.show()
best = df.loc[df['rank1'].idxmax()]
print(f"최고 모델: {best['model']} / {best['rank1']:.2%}")
print(f"85% 게이트: {'통과' if best['rank1'] >= 0.85 else '실패'}")

## 현재 판정

최고는 SigLIP2 Base의 Rank-1 37.89%다. CLIP ViT-L/14는 mean 집계에서 30.53%이며, max/top-5 mean으로 바꾸면 더 낮아졌다. 따라서 gallery 집계만 조정해서 85%를 확보할 수 없다. OSNet도 이번에는 ImageNet 사전학습만 사용했으므로 실제 identity 라벨 기반 ReID fine-tuning과 분리해서 해석한다.

다음 실험은 프로젝트의 여러 identity, 카메라·시간 분리 track, distractor, 속성 검수 라벨을 확보한 뒤 동일 노트북의 입력 계약으로 실행해야 한다.